# Tarea 2 — modelado de alquileres

Comparo tres modelos sobre el dataset unificado: regresión lineal de baseline, HistGradientBoosting y XGBoost. Validación principal con split temporal (cutoff 2025-05-15) y secundaria con KFold estratificado por barrio sobre el ganador. MLflow trackea los tres runs con las métricas, modelo serializado y plots como artifacts en el run del ganador.

In [1]:
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mlflow
import mlflow.sklearn
import mlflow.xgboost

from sklearn.linear_model import Ridge
from sklearn.ensemble import HistGradientBoostingRegressor
from xgboost import XGBRegressor
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, mean_absolute_percentage_error

SEED = 42
np.random.seed(SEED)
OUT = Path('../output')
mlflow.set_tracking_uri(f"file:{(Path('..').resolve() / 'mlruns').as_posix()}")
mlflow.set_experiment('propiedata-alquileres')

2026/05/01 18:54:16 INFO mlflow.tracking.fluent: Experiment with name 'propiedata-alquileres' does not exist. Creating a new experiment.


<Experiment: artifact_location='file:C:/Users/Admin/OneDrive/Desktop/PROYECTOS/Prueba_Tecnica/prueba-tecnica-ds-propiedata/mlruns/647622472006259939', creation_time=1777679656705, experiment_id='647622472006259939', last_update_time=1777679656705, lifecycle_stage='active', name='propiedata-alquileres', tags={}, trace_location=None, workspace='default'>

In [2]:
# cargo el unificado de la Tarea 1
df = pd.read_parquet(OUT / 'dataset_unificado.parquet')
df.shape

(10138, 22)

## features que no dependen del split

In [3]:
# distancia al obelisco como proxy de centralidad + mes y dia de semana
def haversine_km(lat1, lng1, lat2, lng2):
    R = 6371
    lat1, lng1, lat2, lng2 = map(np.radians, [lat1, lng1, lat2, lng2])
    dlat, dlng = lat2 - lat1, lng2 - lng1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlng/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))

OBELISCO = (-34.6037, -58.3816)
df['distancia_centro_km'] = haversine_km(df['lat'], df['lng'], *OBELISCO)
df['mes'] = df['fecha_publicacion'].dt.month.astype('int8')
df['dia_semana'] = df['fecha_publicacion'].dt.dayofweek.astype('int8')

## split temporal

In [4]:
# corte temporal: train = anteriores al cutoff, test = posteriores
CUTOFF = pd.Timestamp('2025-05-15')
train = df[df['fecha_publicacion'] < CUTOFF].copy()
test  = df[df['fecha_publicacion'] >= CUTOFF].copy()
print(f'train={len(train)} | test={len(test)} ({len(test)/len(df):.0%})')
print(f'rango train: {train.fecha_publicacion.min().date()} a {train.fecha_publicacion.max().date()}')
print(f'rango test:  {test.fecha_publicacion.min().date()} a {test.fecha_publicacion.max().date()}')

train=8427 | test=1711 (17%)
rango train: 2024-01-01 a 2025-05-14
rango test:  2025-05-15 a 2025-08-23


## features que se calculan sobre train para evitar leakage

In [5]:
# densidad de listings por barrio: cuantas observaciones de train caen en cada barrio
densidad = train['barrio'].astype(str).value_counts()
train['densidad_barrio'] = train['barrio'].astype(str).map(densidad).astype(float)
test['densidad_barrio']  = test['barrio'].astype(str).map(densidad).fillna(0).astype(float)
train['densidad_barrio'].describe().round(0)

count    8427.0
mean      678.0
std       102.0
min         1.0
25%       691.0
50%       706.0
75%       727.0
max       756.0
Name: densidad_barrio, dtype: float64

In [6]:
# target encoding: precio mediano por (barrio, tipo) calculado solo sobre train
lookup = (train
          .groupby(['barrio', 'tipo_propiedad'], observed=True, as_index=False)['precio_ars_mes']
          .median()
          .rename(columns={'precio_ars_mes': 'precio_mediano_zona'}))
fallback = train['precio_ars_mes'].median()

train = train.merge(lookup, on=['barrio', 'tipo_propiedad'], how='left')
test  = test.merge(lookup,  on=['barrio', 'tipo_propiedad'], how='left')
train['precio_mediano_zona'] = train['precio_mediano_zona'].fillna(fallback)
test['precio_mediano_zona']  = test['precio_mediano_zona'].fillna(fallback)
train['precio_mediano_zona'].describe().round(0)

count       8427.0
mean      847818.0
std       265341.0
min       441900.0
25%       670000.0
50%       797000.0
75%       956500.0
max      2734000.0
Name: precio_mediano_zona, dtype: float64

In [7]:
# defino columnas por tipo (incluyo las features geo nuevas)
TARGET = 'precio_ars_mes'
NUM_COLS  = ['m2_total','m2_cubierto','ambientes','dormitorios','banos','antiguedad_anios','expensas',
             'distancia_centro_km','mes','dia_semana','densidad_barrio','precio_mediano_zona']
CAT_COLS  = ['plataforma','tipo_propiedad','barrio','moneda_origen']
BOOL_COLS = ['cochera','balcon','pileta','flag_missing_expensas','flag_missing_antiguedad','flag_missing_m2_total']
FEATURES  = NUM_COLS + CAT_COLS + BOOL_COLS

def preparar(df_):
    X = df_[FEATURES].copy()
    for c in NUM_COLS:  X[c] = X[c].astype(float)
    for c in BOOL_COLS: X[c] = X[c].astype(float)
    for c in CAT_COLS:  X[c] = X[c].astype(str).fillna('missing')
    return X

X_train, y_train = preparar(train), train[TARGET]
X_test,  y_test  = preparar(test),  test[TARGET]

In [8]:
# helper de metricas (las 4 que pide la consigna)
def evaluar(y_true, y_pred):
    return {
        'r2':   r2_score(y_true, y_pred),
        'rmse': float(np.sqrt(mean_squared_error(y_true, y_pred))),
        'mae':  mean_absolute_error(y_true, y_pred),
        'mape': mean_absolute_percentage_error(y_true, y_pred),
    }

## modelo 1 — Ridge (baseline lineal)

In [9]:
# ridge con OHE para categoricas y mediana+escala para numericas
preproc_lineal = ColumnTransformer([
    ('num', Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler())]), NUM_COLS),
    ('cat', OneHotEncoder(handle_unknown='ignore'), CAT_COLS),
    ('bool', SimpleImputer(strategy='most_frequent'), BOOL_COLS),
])
ridge = Pipeline([('pre', preproc_lineal), ('reg', Ridge(alpha=1.0, random_state=SEED))])

with mlflow.start_run(run_name='ridge'):
    ridge.fit(X_train, y_train)
    pred_ridge = ridge.predict(X_test)
    m_ridge = evaluar(y_test, pred_ridge)
    mlflow.log_params({'modelo': 'Ridge', 'alpha': 1.0, 'cv': 'temporal'})
    mlflow.log_metrics(m_ridge)
    mlflow.sklearn.log_model(ridge, name='model')
    print('ridge:', {k: round(v, 4) for k, v in m_ridge.items()})

2026/05/01 18:54:17 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


ridge: {'r2': 0.7554, 'rmse': 334226.9826, 'mae': 233659.23, 'mape': 0.2731}


## modelo 2 — HistGradientBoosting (ganador)

In [10]:
# HGB maneja NaN nativo y categoricas via parametro
preproc_hgb = ColumnTransformer([
    ('num',  'passthrough', NUM_COLS),
    ('cat',  OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), CAT_COLS),
    ('bool', 'passthrough', BOOL_COLS),
])
cat_idx = list(range(len(NUM_COLS), len(NUM_COLS) + len(CAT_COLS)))
hgb = Pipeline([
    ('pre', preproc_hgb),
    ('reg', HistGradientBoostingRegressor(
        max_iter=500, learning_rate=0.05, max_depth=6,
        categorical_features=cat_idx, random_state=SEED)),
])

with mlflow.start_run(run_name='hgb'):
    hgb.fit(X_train, y_train)
    pred_hgb = hgb.predict(X_test)
    m_hgb = evaluar(y_test, pred_hgb)
    mlflow.log_params({'modelo': 'HistGradientBoosting', 'max_iter': 500, 'lr': 0.05, 'max_depth': 6})
    mlflow.log_metrics(m_hgb)
    mlflow.sklearn.log_model(hgb, name='model')

    # feature importance via permutacion (HGB no expone feature_importances_ directo). n_jobs=1 para evitar lios de joblib en windows
    perm = permutation_importance(hgb, X_test, y_test, n_repeats=5, random_state=SEED, scoring='r2', n_jobs=1)
    fi = pd.Series(perm.importances_mean, index=FEATURES).sort_values(ascending=False)
    fig, ax = plt.subplots(figsize=(7, 6))
    fi.head(15).plot(kind='barh', ax=ax)
    ax.invert_yaxis(); ax.set_title('HGB — permutation importance (top 15)')
    fig.tight_layout(); fig.savefig(OUT / '_fi_hgb.png'); plt.close(fig)
    mlflow.log_artifact(OUT / '_fi_hgb.png')

    # residuos vs precio real
    resid = y_test.values - pred_hgb
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.scatter(y_test, resid, alpha=0.25, s=8)
    ax.axhline(0, color='red', lw=1)
    ax.set_xlabel('precio real (ARS/mes)'); ax.set_ylabel('residuo')
    ax.set_title('HGB — residuos vs precio real')
    fig.tight_layout(); fig.savefig(OUT / '_resid_hgb.png'); plt.close(fig)
    mlflow.log_artifact(OUT / '_resid_hgb.png')

    print('hgb:', {k: round(v, 4) for k, v in m_hgb.items()})

2026/05/01 18:54:26 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


hgb: {'r2': 0.857, 'rmse': 255553.0019, 'mae': 167122.7681, 'mape': 0.1796}


## modelo 3 — XGBoost (con categóricas nativas)

In [11]:
# para xgboost paso categoricas como pd.Categorical asi maneja NaN nativo
def preparar_xgb(df_):
    X = preparar(df_)
    for c in CAT_COLS: X[c] = X[c].astype('category')
    return X

Xtr_xgb, Xte_xgb = preparar_xgb(train), preparar_xgb(test)

In [12]:
# entreno xgb con early stopping
params = dict(n_estimators=800, learning_rate=0.05, max_depth=6,
              subsample=0.85, colsample_bytree=0.85,
              enable_categorical=True, tree_method='hist',
              random_state=SEED, early_stopping_rounds=30)
xgb = XGBRegressor(**params)

with mlflow.start_run(run_name='xgboost'):
    xgb.fit(Xtr_xgb, y_train, eval_set=[(Xte_xgb, y_test)], verbose=False)
    pred_xgb = xgb.predict(Xte_xgb)
    m_xgb = evaluar(y_test, pred_xgb)
    log_params = {k: v for k, v in params.items() if k != 'enable_categorical'}
    log_params['best_iteration'] = int(xgb.best_iteration)
    mlflow.log_params(log_params)
    mlflow.log_metrics(m_xgb)
    mlflow.xgboost.log_model(xgb, name='model')
    print('xgb:', {k: round(v, 4) for k, v in m_xgb.items()})

xgb: {'r2': 0.8154, 'rmse': 290311.9599, 'mae': 184105.9691, 'mape': 0.195}


## comparación

In [13]:
# tabla con las 4 metricas en test
comparacion = pd.DataFrame({'ridge': m_ridge, 'hgb': m_hgb, 'xgb': m_xgb}).T.round(4)
comparacion

,r2,rmse,mae,mape
ridge,0.7554,334226.9826,233659.2300,0.2731
hgb,0.8570,255553.0019,167122.7681,0.1796
xgb,0.8154,290311.9599,184105.9691,0.1950


## CV en train estratificado por barrio (sobre el ganador)

In [14]:
# 5-fold estratificado por barrio: cada fold conserva la distribucion de barrios del train
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
rmses = []
for fold, (tr, va) in enumerate(skf.split(X_train, train['barrio'])):
    m = Pipeline([
        ('pre', preproc_hgb),
        ('reg', HistGradientBoostingRegressor(
            max_iter=500, learning_rate=0.05, max_depth=6,
            categorical_features=cat_idx, random_state=SEED)),
    ])
    m.fit(X_train.iloc[tr], y_train.iloc[tr])
    p = m.predict(X_train.iloc[va])
    rmses.append(float(np.sqrt(mean_squared_error(y_train.iloc[va], p))))
print(f'rmse 5-fold (estratificado por barrio): {np.mean(rmses):,.0f} +/- {np.std(rmses):,.0f}')

rmse 5-fold (estratificado por barrio): 257,436 +/- 8,507


## análisis de errores (sobre el ganador)

In [15]:
# armo un df de test con la prediccion del ganador (HGB) y error porcentual
an = test[['plataforma','tipo_propiedad','barrio','precio_ars_mes']].copy()
an['pred'] = pred_hgb
an['ape']  = (an['pred'] - an['precio_ars_mes']).abs() / an['precio_ars_mes']

In [16]:
# MAPE por cuartil de precio: identifico donde mas falla
an['cuartil_precio'] = pd.qcut(an['precio_ars_mes'], 4, labels=['Q1 (bajo)','Q2','Q3','Q4 (alto)'])
an.groupby('cuartil_precio', observed=True)['ape'].mean().round(3)

cuartil_precio
Q1 (bajo)    0.246
Q2           0.173
Q3           0.140
Q4 (alto)    0.159
Name: ape, dtype: float64

In [17]:
# MAPE por tipo de propiedad
an.groupby('tipo_propiedad', observed=True)['ape'].mean().round(3).sort_values()

tipo_propiedad
depto    0.177
casa     0.186
ph       0.190
Name: ape, dtype: float64

In [18]:
# top 5 barrios donde el modelo se equivoca mas
an.groupby('barrio', observed=True)['ape'].mean().sort_values(ascending=False).head(5).round(3)

barrio
san telmo    0.218
palermo      0.208
flores       0.199
almagro      0.193
nuñez        0.187
Name: ape, dtype: float64

In [19]:
# MAPE por plataforma origen (control de sesgo por fuente)
an.groupby('plataforma', observed=True)['ape'].mean().round(3)

plataforma
A    0.156
B    0.178
C    0.205
Name: ape, dtype: float64

## resumen

Probé los tres modelos con el mismo split temporal (cutoff 2025-05-15) y mismas features. HistGradientBoosting ganó parejo. XGBoost quedó atrás aun con early stopping; con más datos o un tuneo fino probablemente alcanza, pero no en este presupuesto. Ridge sirve solo como piso de comparación.

Las features que más mueven la aguja son `precio_mediano_zona` (target encoding por barrio×tipo) y `m2_total`. Tiene sentido: en alquileres lo que más predice es la zona y el tamaño, todo lo demás es ajuste fino.

Lo que más me llamó la atención del análisis de errores es que el modelo se equivoca más en el cuartil **bajo** de precio, no en el alto. Pasa porque ahí los precios son chicos y cualquier error absoluto se infla porcentualmente. En valor absoluto el cuartil alto sigue siendo el más caro de fittear, pero relativamente el ruido vive abajo.

Por barrio el peor es San Telmo, después Flores y Palermo. San Telmo es chico y heterogéneo; con tan pocas listings ahí adentro el modelo tiene poca señal local. Por plataforma origen C es la peor, lo cual cierra con que tiene la peor calidad de features (mucha info derivada de regex sobre texto libre).

Con más tiempo iría por: target en log para amortiguar la cola, modelos por tipo de propiedad (depto/PH/casa por separado), feature de precio promedio de los k vecinos más cercanos en los últimos 30 días (depende del matching que dejo en la propuesta), y un ensemble HGB+XGB con stacking lineal.